# 02b: Build structured intervals

Run and inspect each stage. Sstructures net-meter evidence; it does not assess inverter conformance.

In [1]:
from __future__ import annotations
import json, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p/'src'/'ausgrid_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the ausgrid_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT/'src'))
from ausgrid_analysis.config import load_config
from ausgrid_analysis.db import connect, site_phase_profile_path, site_profile_path, structured_phase_output_path, structured_site_output_path
from ausgrid_analysis.telemetry_profiles  import build_site_profiles
from ausgrid_analysis.structured_intervals  import build_structured_phase, build_structured_site
from ausgrid_analysis.structured_validation  import validate_structured_telemetry
from ausgrid_analysis.schemas import sql_string
pd.set_option('display.max_columns', 100); plt.style.use('seaborn-v0_8-whitegrid')

## Methodology contract

Before running, review `docs/METHODOLOGY_GATES.md`. 

Load/PV decomposition, battery separation, sign verification, uncurtailed PV and curve comparison are deliberately deferred. 

Candidate inverter phases receive a confidence. Low/unknown sites are not silently promoted.

In [13]:
CONFIG_PATH = PROJECT_ROOT/'analysis.toml'
SAMPLE_MONTH = '2025-04'
SAMPLE_SITE_BUCKET = 0
OVERWRITE_SAMPLE = True
config = load_config(CONFIG_PATH, check_inputs=False)
scope = config.scope(SAMPLE_MONTH, SAMPLE_SITE_BUCKET)
print('Sample scope:', scope.label)
print('Measurement basis: net meter.\nVoltage location: revenue meter.')

Sample scope: month_2025_04__bucket_0_of_32
Measurement basis: net meter.
Voltage location: revenue meter.


## Stage 1: Site/phase profiles and candidate DER-phase mapping

In [14]:
if OVERWRITE_SAMPLE or not (site_phase_profile_path(config, scope).is_file() and site_profile_path(config, scope).is_file()):
    profile_summary = build_site_profiles(config, scope, overwrite=OVERWRITE_SAMPLE)
    display(pd.DataFrame([profile_summary]).T.rename(columns={0:'value'}))
else: print('Reusing existing sample profiles. Set OVERWRITE_SAMPLE=True to rebuild.')
con = connect(config)
phase_profiles = con.execute(f'SELECT * FROM read_parquet({sql_string(site_phase_profile_path(config, scope))}) ORDER BY serial, phase').fetchdf()
site_profiles = con.execute(f'SELECT * FROM read_parquet({sql_string(site_profile_path(config, scope))}) ORDER BY serial').fetchdf()
display(site_profiles.groupby(['analysis_cohort','phase_mapping_confidence'], dropna=False).size().rename('n_sites').reset_index())
display(site_profiles[['serial','analysis_cohort','has_battery','install_phase_count','power_available_phases','inferred_der_phases','phase_mapping_method','phase_mapping_confidence','solar_only_mapped_cohort']].head(30))
display(phase_profiles.groupby('phase').agg(n_pairs=('serial','size'), power_missing=('power_measurement_available',lambda x:(~x).sum()), zero_voltage=('n_voltage_at_or_below_zero','sum'), median_signature=('solar_signature_w','median')))
assert site_profiles.serial.is_unique
assert not site_profiles.formal_inverter_conformance_assessable.any()
print('Stage 1 structural gate passed. Review low/unknown mappings before continuing.')

2026-08-04 16:26:22,734 | INFO | Structured telemetry profiles written: 42 sites


,value
created_utc,2026-08-04T06:26:22.732940+00:00
scope,month_2025_04__bucket_0_of_32
site_phase_rows,89
sites,42
mapping_confidence,"{'high': 24, 'insufficient': 17, 'unknown': 1}"
primary_cohort_sites,18
battery_sites,12
site_phase_profile,C:\Users\z3553082\OneDrive - UNSW\Documents\CI...
site_profile,C:\Users\z3553082\OneDrive - UNSW\Documents\CI...


,analysis_cohort,phase_mapping_confidence,n_sites
0,solar_battery,high,6
1,solar_battery,insufficient,6
2,solar_only,high,18
3,solar_only,insufficient,11
4,NaN,unknown,1


,serial,analysis_cohort,has_battery,install_phase_count,power_available_phases,inferred_der_phases,phase_mapping_method,phase_mapping_confidence,solar_only_mapped_cohort
0,810140828,solar_only,False,1.0,A|B|C,B,ranked_local_day_night_export_signature,high,True
1,810208902,solar_battery,True,3.0,A|B|C,A|B|C,all_power_available_phases,high,False
2,810244040,solar_only,False,1.0,,,insufficient_power_phases,insufficient,False
3,810405327,solar_battery,True,3.0,A|B|C,A|B|C,all_power_available_phases,high,False
4,810564991,solar_only,False,1.0,,,insufficient_power_phases,insufficient,False
5,810616092,solar_only,False,1.0,A|B,A,ranked_local_day_night_export_signature,high,True
6,810667645,solar_only,False,3.0,A|B|C,A|B|C,all_power_available_phases,high,True
7,810689648,solar_only,False,1.0,,,insufficient_power_phases,insufficient,False
8,810694368,solar_only,False,3.0,A|B|C,A|B|C,all_power_available_phases,high,True
9,810787019,solar_only,False,1.0,A|B|C,A,ranked_local_day_night_export_signature,high,True


,n_pairs,power_missing,zero_voltage,median_signature
phase,,,,
A,42,17,0.0,1198.848966
B,25,0,0.0,1317.857589
C,22,0,0.0,1075.493948


Stage 1 structural gate passed. Review low/unknown mappings before continuing.


In [15]:
mapping_counts = (
    site_profiles
    .groupby(
        [
            "install_phase_count",
            "phase_mapping_method",
            "phase_mapping_confidence",
        ],
        dropna=False,
    )
    .size()
    .rename("n_sites")
    .reset_index()
)

display(mapping_counts)

,install_phase_count,phase_mapping_method,phase_mapping_confidence,n_sites
0,1.0,insufficient_power_phases,insufficient,16
1,1.0,ranked_local_day_night_export_signature,high,7
2,3.0,all_power_available_phases,high,17
3,3.0,insufficient_power_phases,insufficient,1
4,NaN,missing_install_phase_count,unknown,1


In [16]:
power_missingness = (
    phase_profiles
    .groupby("phase")
    .agg(
        n_site_phase_pairs=("serial", "size"),
        active_power_always_null_sites=(
            "active_power_always_null",
            "sum",
        ),
        reactive_power_always_null_sites=(
            "reactive_power_always_null",
            "sum",
        ),
        p_q_mismatch_sites=(
            "n_p_q_missingness_mismatch",
            lambda values: (values > 0).sum(),
        ),
    )
    .reset_index()
)

display(power_missingness)

,phase,n_site_phase_pairs,active_power_always_null_sites,reactive_power_always_null_sites,p_q_mismatch_sites
0,A,42,17,17,0
1,B,25,0,0,0
2,C,22,0,0,0


In [17]:
insufficient_sites = site_profiles.loc[
    site_profiles["phase_mapping_method"].eq(
        "insufficient_power_phases"
    ),
    [
        "serial",
        "analysis_cohort",
        "install_phase_count",
        "observed_phases",
        "power_available_phases",
        "inferred_der_phases",
        "phase_mapping_confidence",
    ],
].sort_values(
    ["install_phase_count", "serial"]
)

display(insufficient_sites)
print("Insufficient sites:", len(insufficient_sites))

,serial,analysis_cohort,install_phase_count,observed_phases,power_available_phases,inferred_der_phases,phase_mapping_confidence
2,810244040,solar_only,1.0,A,,,insufficient
4,810564991,solar_only,1.0,A,,,insufficient
7,810689648,solar_only,1.0,A,,,insufficient
11,810819331,solar_only,1.0,A,,,insufficient
14,810847375,solar_battery,1.0,A,,,insufficient
15,810851304,solar_only,1.0,A,,,insufficient
23,810908167,solar_battery,1.0,A,,,insufficient
26,810944085,solar_battery,1.0,A,,,insufficient
27,810944091,solar_battery,1.0,A,,,insufficient
28,810948215,solar_only,1.0,A,,,insufficient


Insufficient sites: 17


## Stage 2: Structured phase intervals

In [18]:
phase_out = structured_phase_output_path(config, scope)
if OVERWRITE_SAMPLE or not phase_out.is_dir():
    display(pd.DataFrame([build_structured_phase(config, scope, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0:'value'}))
else: print('Reusing existing structured phase sample.')
phase_glob = str(phase_out/'**'/'*.parquet')
phase_check = con.execute(f'''SELECT count(*) n_rows, count(DISTINCT serial) n_sites,
 count_if(is_inferred_der_phase) inferred_der_phase_rows,
 count_if(NOT voltage_valid_for_analysis) invalid_voltage_rows,
 count_if(NOT power_measurement_available) missing_power_rows,
 count_if(utc_offset_minutes=600) aest_rows, count_if(utc_offset_minutes=660) aedt_rows
 FROM read_parquet({sql_string(phase_glob)}, hive_partitioning=true)''').fetchdf()
display(phase_check)
display(con.execute(f'SELECT * FROM read_parquet({sql_string(phase_glob)}, hive_partitioning=true) ORDER BY timestamp_utc, serial, phase LIMIT 20').fetchdf())

,value
created_utc,2026-08-04T06:26:25.779180+00:00
scope,month_2025_04__bucket_0_of_32
rows,759287
sites,42
rows_on_inferred_der_phases,493630
output,C:\Users\z3553082\OneDrive - UNSW\Documents\CI...


,n_rows,n_sites,inferred_der_phase_rows,invalid_voltage_rows,missing_power_rows,aest_rows,aedt_rows
0,759287,42,493630.0,0.0,145297.0,639671.0,119616.0


,serial,timestamp_utc,timestamp_local,phase,voltage_v,current_a,active_power_raw_w,reactive_power_raw_var,p_export_w,q_absorbing_var,q_generator_var,source_month,source_file,duplicate_count,duplicate_source_file_count,duplicate_status,row_has_null_measurement,voltage_physical_ok,metadata_available,analysis_cohort,install_phase_count,voltage_valid_for_analysis,power_measurement_available,current_measurement_available,is_inferred_der_phase,local_date,local_hour,local_minute,utc_offset_minutes,has_battery,phase_mapping_method,phase_mapping_confidence,phase_mapping_assessable,solar_only_mapped_cohort,measurement_basis,voltage_measurement_location,formal_inverter_conformance_assessable,month_utc,site_bucket,year_utc
0,810140828,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,A,243.442017,0.332560,35.736385,-25.525990,-35.736385,-25.525990,25.525990,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_only,1,True,True,True,False,2025-04-01,11,0,660,False,ranked_local_day_night_export_signature,high,True,True,net_meter,revenue_meter,False,4,0,2025
1,810140828,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,B,241.494064,7.150036,-1546.875000,684.096500,1546.875000,684.096500,-684.096500,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_only,1,True,True,True,True,2025-04-01,11,0,660,False,ranked_local_day_night_export_signature,high,True,True,net_meter,revenue_meter,False,4,0,2025
2,810140828,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,C,241.947556,0.396804,71.472770,-56.157177,-71.472770,-56.157177,56.157177,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_only,1,True,True,True,False,2025-04-01,11,0,660,False,ranked_local_day_night_export_signature,high,True,True,net_meter,revenue_meter,False,4,0,2025
3,810208902,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,A,238.010437,3.431412,-270.575500,-740.253700,270.575500,-740.253700,740.253700,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_battery,3,True,True,True,True,2025-04-01,11,0,660,True,all_power_available_phases,high,True,False,net_meter,revenue_meter,False,4,0,2025
4,810208902,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,B,238.546371,2.989259,-704.517334,-112.314354,704.517334,-112.314354,112.314354,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_battery,3,True,True,True,True,2025-04-01,11,0,660,True,all_power_available_phases,high,True,False,net_meter,revenue_meter,False,4,0,2025
5,810208902,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,C,237.793991,3.654379,-852.568054,-148.050735,852.568054,-148.050735,148.050735,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_battery,3,True,True,True,True,2025-04-01,11,0,660,True,all_power_available_phases,high,True,False,net_meter,revenue_meter,False,4,0,2025
6,810244040,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,A,240.040833,16.049790,NaN,NaN,NaN,NaN,NaN,2025-04,jm_unswnmisapr25,1,1,unique,True,True,True,solar_only,1,True,False,True,<NA>,2025-04-01,11,0,660,False,insufficient_power_phases,insufficient,False,False,net_meter,revenue_meter,False,4,0,2025
7,810405327,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,A,238.824661,2.373268,-479.888600,-10.210396,479.888600,-10.210396,10.210396,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_battery,3,True,True,True,True,2025-04-01,11,0,660,True,all_power_available_phases,high,True,False,net_meter,revenue_meter,False,4,0,2025
8,810405327,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,B,238.618515,0.506398,35.736385,-107.209152,-35.736385,-107.209152,107.209152,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_battery,3,True,True,True,True,2025-04-01,11,0,660,True,all_power_available_phases,high,True,False,net_meter,revenue_meter,False,4,0,2025
9,810405327,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,C,237.361115,4.667174,418.626221,-239.944300,-418.626221,-239.944300,239.944300,2025-04,jm_unswnmisapr25,1,1,unique,False,True,True,solar_battery,3,True,True,True,True,2025-04-01,11,0,660,True,all_power_available_phases,high,True,False,n

## Stage 3: Structured site intervals and safe aggregation

In [19]:
site_out = structured_site_output_path(config, scope)
if OVERWRITE_SAMPLE or not site_out.is_dir():
    display(pd.DataFrame([build_structured_site(config, scope, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0:'value'}))
else: print('Reusing existing structured site sample.')
site_glob = str(site_out/'**'/'*.parquet')
site_preview = con.execute(f'''SELECT * FROM read_parquet({sql_string(site_glob)}, hive_partitioning=true)
 ORDER BY timestamp_utc, serial LIMIT 1000''').fetchdf()
display(site_preview.head(20))
display(site_preview.groupby(['analysis_cohort','has_battery','phase_mapping_confidence'], dropna=False).agg(rows=('serial','size'), complete=('der_phase_power_complete','sum')).reset_index())
assert site_preview.loc[~site_preview.der_phase_power_complete, 'p_export_der_phase_net_complete_w'].isna().all()

2026-08-04 16:26:27,751 | INFO | Structured site intervals written: 358547 rows


,value
created_utc,2026-08-04T06:26:27.751478+00:00
scope,month_2025_04__bucket_0_of_32
rows,358547
sites,42
incomplete_der_power_rows,0
output,C:\Users\z3553082\OneDrive - UNSW\Documents\CI...


,serial,timestamp_utc,timestamp_local,local_date,local_hour,local_minute,utc_offset_minutes,metadata_available,analysis_cohort,has_battery,phase_mapping_method,phase_mapping_confidence,phase_mapping_assessable,solar_only_mapped_cohort,observed_phase_rows,measured_power_phase_rows,expected_der_phase_rows,measured_der_phase_rows,voltage_min_valid_v,voltage_mean_valid_v,voltage_max_valid_v,der_voltage_min_valid_v,der_voltage_mean_valid_v,der_voltage_max_valid_v,p_export_net_observed_w,q_absorbing_net_observed_var,p_export_der_phase_net_observed_w,q_absorbing_der_phase_net_observed_var,voltage_a_v,voltage_b_v,voltage_c_v,p_export_a_w,p_export_b_w,p_export_c_w,der_phase_power_complete,p_export_der_phase_net_complete_w,q_absorbing_der_phase_net_complete_var,measurement_basis,voltage_measurement_location,formal_inverter_conformance_assessable,month_utc,site_bucket,year_utc
0,810140828,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,ranked_local_day_night_export_signature,high,True,True,3,3.0,1.0,1.0,241.494064,242.294546,243.442017,241.494064,241.494064,241.494064,1439.665845,602.413333,1546.875000,684.096500,243.442017,241.494064,241.947556,-35.736385,1546.875000,-71.472770,True,1546.875000,684.096500,net_meter,revenue_meter,False,4,0,2025
1,810208902,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_battery,True,all_power_available_phases,high,True,False,3,3.0,3.0,3.0,237.793991,238.116933,238.546371,237.793991,238.116933,238.546371,1827.660888,-1000.618789,1827.660888,-1000.618789,238.010437,238.546371,237.793991,270.575500,704.517334,852.568054,True,1827.660888,-1000.618789,net_meter,revenue_meter,False,4,0,2025
2,810244040,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,insufficient_power_phases,insufficient,False,False,1,0.0,NaN,0.0,240.040833,240.040833,240.040833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,240.040833,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,net_meter,revenue_meter,False,4,0,2025
3,810405327,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_battery,True,all_power_available_phases,high,True,False,3,3.0,3.0,3.0,237.361115,238.268097,238.824661,237.361115,238.268097,238.824661,25.525994,-357.363848,25.525994,-357.363848,238.824661,238.618515,237.361115,479.888600,-35.736385,-418.626221,True,25.525994,-357.363848,net_meter,revenue_meter,False,4,0,2025
4,810564991,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,insufficient_power_phases,insufficient,False,False,1,0.0,NaN,0.0,244.513900,244.513900,244.513900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,244.513900,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,net_meter,revenue_meter,False,4,0,2025
5,810616092,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,ranked_local_day_night_export_signature,high,True,True,2,2.0,1.0,1.0,239.257538,239.257538,239.257538,239.257538,239.257538,239.257538,-4885.674320,-484.993800,-4885.674320,-484.993800,239.257538,239.257538,NaN,-4885.674320,0.000000,NaN,True,-4885.674320,-484.993800,net_meter,revenue_meter,False,4,0,2025
6,810667645,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,all_power_available_phases,high,True,True,3,3.0,3.0,3.0,244.318069,245.802226,246.781357,244.318069,245.802226,246.781357,4278.155854,1112.933173,4278.155854,1112.933173,246.307251,244.318069,246.781357,-10.210396,4410.891000,-122.524750,True,4278.155854,1112.933173,net_meter,revenue_meter,False,4,0,2025
7,810689648,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,insufficient_power_phases,insufficient,False,False,1,0.0,NaN,0.0,243.957336,243.957336,243.957336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,243.957336,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,net_meter,revenue_meter,False,4,0,2025
8,810694368,2025-04-01 00:00:00+00:00,2025-04-01 11:00:00,2025-04-01,11,0,660,True,solar_only,False,all_power_available_phases,high,True,True,3,3.0,3.0,3.0

,analysis_cohort,has_battery,phase_mapping_confidence,rows,complete
0,solar_battery,True,high,143,143
1,solar_battery,True,insufficient,143,0
2,solar_only,False,high,429,429
3,solar_only,False,insufficient,261,0
4,NaN,False,unknown,24,0


## Sample validation gate

In [9]:
validation = validate_structured_telemetry(config, scope)
display(pd.DataFrame([{k:v for k,v in validation.items() if k not in {'monthly_coverage','methodology_state','failures'}}]).T.rename(columns={0:'value'}))
display(pd.DataFrame(validation['monthly_coverage']))
display(pd.DataFrame([validation['methodology_state']]).T.rename(columns={0:'state'}))
assert validation['status']=='pass', validation['failures']
print('Structured telemetry sample passed. Interpret mappings scientifically before unlocking full build.')

,value
created_utc,2026-08-04T06:23:52.360706+00:00
scope,month_2025_04__bucket_0_of_32
status,pass
canonical_rows,759287
canonical_site_time_keys,358547
canonical_sites,42
structured_phase_rows,759287
structured_phase_duplicate_keys,0
structured_site_rows,358547
structured_site_duplicate_keys,0


,year_utc,month_utc,n_site_intervals,n_sites,n_complete_der_power
0,2025,4,358547,42,204610.0


,state
load_pv_decomposition,not_yet_performed
battery_handling,flagged_and_excluded_from_primary_cohort
sign_convention,working_assumption_not_verified
phase_mapping,candidate_mapping_with_confidence
timezone,UTC_and_Australia/Sydney_retained
voltage_location,revenue_meter_not_corrected
uncurtailed_pv,not_yet_estimated
comparison_basis,net_meter_only_no_formal_curve_comparison


Structured telemetry sample passed. Interpret mappings scientifically before unlocking full build.


In [10]:
display(
    site_profiles.groupby(
        [
            "install_phase_count",
            "phase_mapping_method",
            "phase_mapping_confidence",
        ],
        dropna=False,
    )
    .size()
    .rename("n_sites")
    .reset_index()
)

,install_phase_count,phase_mapping_method,phase_mapping_confidence,n_sites
0,1.0,insufficient_power_phases,insufficient,16
1,1.0,ranked_local_day_night_export_signature,high,7
2,3.0,all_power_available_phases,high,17
3,3.0,insufficient_power_phases,insufficient,1
4,NaN,missing_install_phase_count,unknown,1


## Full dataset build (locked)

This can create another large phase-level dataset. Run only after reviewing the complete sample.

In [ ]:
FULL_RUN_CONFIRMATION = 'RUN STRUCTURED TELEMETRY FULL'  # Change to: RUN STRUCTURED TELEMETRY FULL
OVERWRITE_FULL = True
assert FULL_RUN_CONFIRMATION == 'RUN STRUCTURED TELEMETRY FULL', 'Full run remains locked.'
full_scope = config.scope(None, None)
print('Full run unlocked:', full_scope.label)

Full run unlocked: full


In [21]:
display(pd.DataFrame([build_site_profiles(config, full_scope, overwrite=OVERWRITE_FULL)]).T.rename(columns={0:'value'}))
display(pd.DataFrame([build_structured_phase(config, full_scope, overwrite=OVERWRITE_FULL)]).T.rename(columns={0:'value'}))
display(pd.DataFrame([build_structured_site(config, full_scope, overwrite=OVERWRITE_FULL)]).T.rename(columns={0:'value'}))
full_validation = validate_structured_telemetry(config, full_scope)
display(pd.DataFrame([{k:v for k,v in full_validation.items() if k!='monthly_coverage'}]).T.rename(columns={0:'value'}))
display(pd.DataFrame(full_validation['monthly_coverage']))
assert full_validation['status']=='pass', full_validation['failures']
print('Structured telemetry full build passed.')
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-08-04 16:39:13,531 | INFO | Structured telemetry profiles written: 1342 sites


,value
created_utc,2026-08-04T06:39:13.529311+00:00
scope,full
site_phase_rows,2724
sites,1342
mapping_confidence,"{'high': 636, 'insufficient': 614, 'unknown': ..."
primary_cohort_sites,453
battery_sites,355
site_phase_profile,C:\Users\z3553082\OneDrive - UNSW\Documents\CI...
site_profile,C:\Users\z3553082\OneDrive - UNSW\Documents\CI...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,value
created_utc,2026-08-04T08:31:06.677856+00:00
scope,full
status,pass
failures,[]
canonical_rows,284087739
canonical_site_time_keys,140069836
canonical_sites,1342
structured_phase_rows,284087739
structured_phase_duplicate_keys,0
structured_site_rows,140069836


,year_utc,month_utc,n_site_intervals,n_sites,n_complete_der_power
0,2024,8,11839849,1327,5956348.0
1,2024,9,11450002,1327,5753054.0
2,2024,10,11881243,1338,5934280.0
3,2024,11,11557950,1338,5743230.0
4,2024,12,11928895,1338,5928671.0
5,2025,1,11927780,1336,5928164.0
6,2025,2,10773504,1336,5354496.0
7,2025,3,11927791,1336,5928175.0
8,2025,4,11384175,1336,5654807.0
9,2025,5,11927808,1336,5928192.0


Structured telemetry full build passed.
